In [70]:
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import numpy as np
from sklearn.manifold import TSNE
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
import os



In [71]:
def load_and_combine_model_datasets(base_path: str, model_names: list[str]) -> pd.DataFrame:
    """
    Load CSV files for each model and combine them into a single DataFrame with model labels.
    
    Args:
        base_path: Base directory path containing the CSV files
        model_names: List of model names to process
        
    Returns:
        Combined DataFrame with generator model labels
    """
    all_dfs = []
    
    for model in model_names:
        file_path = os.path.join(base_path, f"{model}_dataset.csv")
        try:
            df = pd.read_csv(file_path, index_col=0)
            # Add generator model name as a column
            df['generator_model'] = model
            all_dfs.append(df)
        except FileNotFoundError:
            print(f"Warning: Could not find dataset for {model} at {file_path}")
    
    # Combine all dataframes
    combined_df = pd.concat(all_dfs, ignore_index=True)
    return combined_df

# Load and combine all datasets
base_path = "./"
model_names = ['gpt-4o', 'deepseek-v3', 'llama_8b', 'llama_70b', 'o1_mini', 'sonnet', 'gpt-4o-mini']

combined_df = load_and_combine_model_datasets(base_path, model_names)

In [72]:
combined_df.generator_model.value_counts()

generator_model
gpt-4o         210
o1_mini        210
deepseek-v3    201
llama_70b      201
sonnet         198
llama_8b       186
gpt-4o-mini    186
Name: count, dtype: int64

In [73]:
logical_and_df = combined_df[combined_df.question_type == "P_and_Q"]
logical_and_df.head()

,question_triple_id,iteration,hypothesis,topic,reasoning_flaw,question_type,question_title,question_body,avg_forecast,individual_forecasts,consistency_score,generation_reasoning,generator_model
2,iter4_h2_q1,4,The model overestimates the impact of social m...,Sustainable Fashion,NaN,P_and_Q,Will Patagonia's sales of environmentally-frie...,This question resolves as YES if both conditio...,0.75,"[0.75, 0.75, 0.75, 0.75, 0.75]",0.597867,NaN,gpt-4o
3,iter13_h2_q0,13,The model underestimates the impact of geopoli...,Energy and Geopolitics,NaN,P_and_Q,Will China increase its investment in renewabl...,This question resolves as YES if both conditio...,0.41,"[0.45, 0.35, 0.35, 0.45, 0.45]",0.558942,NaN,gpt-4o
6,iter7_h4_q0,7,The model underestimates the impact of geopoli...,Global Energy Markets,NaN,P_and_Q,Will China increase its investment in renewabl...,This question resolves as YES if both conditio...,0.45,"[0.45, 0.45, 0.45, 0.45, 0.45]",0.544915,NaN,gpt-4o
10,iter7_h1_q1,7,The model overestimates the impact of social m...,Sustainable Fashion,NaN,P_and_Q,Will Patagonia's sales of environmentally-frie...,This question resolves as YES if both conditio...,0.57,"[0.55, 0.55, 0.65, 0.45, 0.65]",0.540032,NaN,gpt-4o
12,iter9_h1_q0,9,The model underestimates the impact of geopoli...,Geopolitics and Energy,NaN,P_and_Q,Will China increase its investment in renewabl...,This question resolves as YES if both conditio...,0.43,"[0.35, 0.45, 0.45, 0.45, 0.45]",0.518487,NaN,gpt-4o


In [74]:
# %pip install --upgrade huggingface_hub==0.27.0 datasets==3.2.0

In [75]:
# # Define topic groupings
# topic_to_group = {
#     # Energy and Environment
#     'energy storage': 'Energy & Environment',
#     'economics and energy': 'Energy & Environment',
#     'energy': 'Energy & Environment',
#     'energy and environment': 'Energy & Environment',
#     'energy policy and sustainability': 'Energy & Environment',
    
#     # Economics
#     'economics': 'Economics',
#     'economics and finance': 'Economics',
    
#     # Politics
#     'geopolitics': 'Politics',
#     'economics and education': 'Politics',
#     'geopolitics and national security': 'Politics',
#     'labor market and technology': 'Politics',
    
#     # Computing
#     'technology and regulation': 'AI & Computer Science',
#     'materials science and quantum computing':  'AI & Computer Science',
#     'cryptography':  'AI & Computer Science',
#     'ai and computing':  'AI & Computer Science',
#     'artificial general intelligence':  'AI & Computer Science',
#     'robotics':  'AI & Computer Science',
#     'ai and robotics':  'AI & Computer Science',
#     'cybersecurity':  'AI & Computer Science',
    
#     #Social Media
#     'social media and politics': 'Media',
#     'social media': 'Media',
#     'social media and regulation': 'Media',
#     'social media and mental health': 'Media',
#     'election misinformation': 'Media',
#     'social media regulation':  'Media',
#     'marketing and e-commerce': 'Media',
#     'elections and misinformation': 'Media',
    
#     # Sports
#     'sports': 'Sports',
#     'sports and olympics': 'Sports'
# }

# def analyze_model_topics(logical_and_df: pd.DataFrame, model_names: list[str]):
#     """
#     Analyze and visualize topic distribution between models using MPNet embeddings and t-SNE.
#     Uses predefined topic groupings for consistent categorization.
#     """
#     # Filter for specified models
#     model_df = logical_and_df[logical_and_df['generator_model'].isin(model_names)]
    
#     # Initialize MPNet model
#     mpnet = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
    
#     # Get embeddings for questions
#     old_texts = model_df['question_body'].tolist()
#     embeddings = mpnet.encode(old_texts, show_progress_bar=True)
#     texts = []
#     for text in old_texts:
#         formatted = ""
#         for i in range(0, len(text), 80):
#             chunk = text[i:i+80]
#             formatted += chunk
#             if i + 80 < len(text):
#                 formatted += "<br>"
#         texts.append(formatted)
    
#     # t-SNE dimensionality reduction
#     tsne = TSNE(n_components=2, perplexity=30, random_state=42)
#     tsne_embeddings = tsne.fit_transform(embeddings)
    
#     # Map topics to groups
#     def get_topic_group(topic):
#         if pd.isna(topic):
#             return 'Other'
#         return topic_to_group.get(topic.lower(), 'Other')
    
#     # Create visualization DataFrame
#     viz_df = pd.DataFrame({
#         'TSNE1': tsne_embeddings[:, 0],
#         'TSNE2': tsne_embeddings[:, 1],
#         'Model': model_df['generator_model'].values,
#         'Question': texts,
#         'Question_Preview': [t[:100] + '...' for t in texts],
#         'Original_Topic': model_df['topic'].values,
#         'Topic_Group': model_df['topic'].apply(get_topic_group)
#     })
    
#     # Define color scheme
# # ... existing code ...

#     # Define color scheme
#     color_map = {
#         'Energy & Environment': '#2ecc71',    # Green
#         'Economics': '#e74c3c',               # Red
#         'Politics': '#3498db',                # Blue
#         'Computing': '#9b59b6',               # Purple
#         'Artificial General Intelligence': '#f1c40f',  # Yellow
#         'Biotechnology': '#1abc9c',           # Turquoise
#         'Sports': '#e67e22',                  # Orange
#         'Other': 'rgba(149, 165, 166, 0.5)'  # Semi-transparent grey
#     }

# # ... existing code ...    
#         # Create interactive plot
#     fig = px.scatter(
#         viz_df,
#         x='TSNE1',
#         y='TSNE2',
#         color='Topic_Group',
#         color_discrete_map=color_map,
#         hover_data={
#             'TSNE1': False,
#             'TSNE2': False,
#             'Model': True,
#             'Original_Topic': True,
#             'Topic_Group': True,
#             'Question_Preview': True
#         },
#         title='Topic Distribution of Adaptive Questions Generated for DeepSeek-V3',
#         height=800,
#         width=2000  # Compressed width
#     )
    
#     # Update layout
#     fig.update_traces(marker=dict(size=15))
#     fig.update_layout(
#         template='plotly_white',
#         legend=dict(
#             title=dict(
#                 text='Question Topics',
#                 font=dict(size=32)  # Increased legend title font size
#             ),
#             yanchor="top",
#             y=1.0,
#             xanchor="left",
#             x=0.02,
#             bgcolor='#fdfbf9',
#             font=dict(size=29),  # Increased legend text font size
#             bordercolor="Black",
#             borderwidth=0
#         ),
#         title=dict(
#             text='', #Topic Distribution of Adaptive Questions Generated for DeepSeek-V3
#             x=0.35,        # Center title horizontally
#             y=0.95,       # Adjust vertical position
#             xanchor='center',  # Anchor point for centering
#             yanchor='top',    # Anchor point for vertical position
#             font=dict(size=20)
#         ),
#         xaxis=dict(
#             title='TSNE1',
#             title_font=dict(size=38),  # Increased axis title font size
#             tickfont=dict(size=1)     # Increased axis tick font size
#         ),
#         yaxis=dict(
#             title='TSNE2',
#             title_font=dict(size=38),  # Increased axis title font size
#             tickfont=dict(size=1)     # Increased axis tick font size
#         ),
#         margin=dict(l=20, r=100, t=50, b=20),
#         plot_bgcolor='#fdfbf9'
#     )
    
#     # Show plot
#     fig.show()
#     fig.write_image("deepseek_v3_topic_distribution.png", 
#                     engine="kaleido",
#                     scale=4.0,  # Increases resolution by 4x
#                     width=2000,  # Base width
#                     height=800)  # Base height
    
#     # Print topic distribution
#     print("\nTopic group distribution by model:")
#     for model in model_names:
#         model_data = viz_df[viz_df['Model'] == model]
#         topic_dist = model_data['Topic_Group'].value_counts()
#         print(f"\n{model} topic distribution:")
#         print(topic_dist)
    
#     return viz_df

# # Run the analysis
# model_names = ['deepseek-v3']
# viz_df = analyze_model_topics(logical_and_df, model_names)

In [76]:
# load in the base huggingface dataset
from datasets import load_dataset

# ds_base = load_dataset("prithvi3/cond_100_test")
# ds_base['test']

# Ps, Qs, PQs = [item['P_body'] for item in ds_base['test']], [item['Q_body'] for item in ds_base['test']], [item['P_and_Q_body'] for item in ds_base['test']]
# # all_questions = [q for triple in zip(Ps, Qs, PQs) for q in triple]

# all_questions = PQs # Ps + Qs

ds_base = load_dataset("prithvi3/consistency_250_test")
PQs = [item['P_and_Q_body'] for item in ds_base['test']]
all_questions = PQs


In [77]:
all_questions[0]

'This question will resolve as **Yes** if both Czechia and Armenia participate in their respective Semi-finals and are officially announced by the European Broadcasting Union (EBU) as having qualified for the Eurovision 2024 Grand Final. If either Czechia or Armenia does not qualify, the question will resolve as **No**.'

In [ ]:
def analyze_model_topics_with_unlabeled(
    logical_and_df: pd.DataFrame,
    model_names: list[str],
    all_questions: list[str]
) -> pd.DataFrame:
    """
    Analyze and visualize topic distribution between models (with known topics)
    plus additional unlabeled questions using MPNet embeddings and t-SNE.
    The unlabeled questions are plotted in gray with alpha=0.5,
    but are excluded from the legend.
    """
    import pandas as pd
    import plotly.express as px
    import numpy as np
    from sentence_transformers import SentenceTransformer
    from sklearn.manifold import TSNE
    
    # Filter for specified models
    labeled_df = logical_and_df[logical_and_df['generator_model'].isin(model_names)].copy()
    
    # Build a DataFrame for the unlabeled questions
    unlabeled_df = pd.DataFrame({
        'question_body': all_questions,
        'generator_model': ['Unlabeled'] * len(all_questions),
        'topic': [np.nan] * len(all_questions)
    })
    
    # Combine labeled and unlabeled for a single embedding pass
    combined_df = pd.concat([labeled_df, unlabeled_df], ignore_index=True)
    
    # Initialize MPNet model
    mpnet = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
    
    # Get embeddings for combined questions
    old_texts = combined_df['question_body'].tolist()
    embeddings = mpnet.encode(old_texts, show_progress_bar=True)
    # Add line breaks every 80 chars for plotly hover readability
    texts = []
    for text in old_texts:
        formatted = ""
        for i in range(0, len(text), 80):
            chunk = text[i:i+80]
            formatted += chunk
            if i + 80 < len(text):
                formatted += "<br>"
        texts.append(formatted)
    
    # t-SNE dimensionality reduction
    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    tsne_embeddings = tsne.fit_transform(embeddings)
    
    # Define topic group mapping
    topic_to_group = {
        # Energy & Environment
        'energy storage': 'Energy & Environment',
        'economics and energy': 'Energy & Environment',
        'energy': 'Energy & Environment',
        'energy and environment': 'Energy & Environment',
        'energy policy and sustainability': 'Energy & Environment',
        
        # Economics
        'economics': 'Economics',
        'economics and finance': 'Economics',
        
        # Politics
        'geopolitics': 'Politics',
        'economics and education': 'Politics',
        'geopolitics and national security': 'Politics',
        'labor market and technology': 'Politics',
        
        # Computing
        'technology and regulation': 'AI & Computer Science',
        'materials science and quantum computing': 'AI & Computer Science',
        'cryptography': 'AI & Computer Science',
        'ai and computing': 'AI & Computer Science',
        'artificial general intelligence': 'AI & Computer Science',
        'robotics': 'AI & Computer Science',
        'ai and robotics': 'AI & Computer Science',
        'cybersecurity': 'AI & Computer Science',
        
        # Social Media
        'social media and politics': 'Media',
        'social media': 'Media',
        'social media and regulation': 'Media',
        'social media and mental health': 'Media',
        'election misinformation': 'Media',
        'social media regulation': 'Media',
        'marketing and e-commerce': 'Media',
        'elections and misinformation': 'Media',
        
        # Sports
        'sports': 'Sports',
        'sports and olympics': 'Sports'
    }
    
    def get_topic_group(topic: str) -> str:
        if pd.isna(topic):
            return 'Unlabeled'
        return topic_to_group.get(topic.lower(), 'Other')
    
    # Construct visualization DataFrame
    combined_viz = pd.DataFrame({
        'TSNE1': tsne_embeddings[:, 0],
        'TSNE2': tsne_embeddings[:, 1],
        'Model': combined_df['generator_model'].values,
        'Question': texts,
        'Topic_Group': combined_df['topic'].apply(get_topic_group)
    })
    
    # Define color map (unlabeled in semi-transparent gray)
    color_map = {
        'Energy & Environment': '#2ecc71',
        'Economics': '#e74c3c',
        'Politics': '#3498db',
        'AI & Computer Science': '#9b59b6',
        'Other': '#95a5a6',
        'Unlabeled': 'rgba(128, 128, 128, 0.2)'
    }
    
    # Create scatter plot
    fig = px.scatter(
        combined_viz,
        x='TSNE1',
        y='TSNE2',
        color='Topic_Group',
        color_discrete_map=color_map,
        hover_data={
            'TSNE1': False,
            'TSNE2': False,
            'Model': True,
            'Topic_Group': True,
            'Question': True
        },
        title='Topic Distribution (with Unlabeled)',
        height=800,
        width=2000
    )
    fig.update_traces(marker=dict(size=15))
    
    # Hide "Unlabeled" in the legend
    for trace in fig.data:
        if trace.name == 'Unlabeled':
            # trace.showlegend = False
            trace.name = 'Static Questions'
        # if trace.name == 'Other':
        #     trace.showlegend = False
    
    # General layout updates
    fig.update_layout(
        template='plotly_white',
        legend=dict(
            title=dict(
                text='Question Topics',
                font=dict(size=32)
            ),
            yanchor="top",
            y=1.0,
            xanchor="right", 
            x=0.98,
            bgcolor='#fdfbf9',
            font=dict(size=29),
            bordercolor="Black",
            borderwidth=0,
            traceorder='normal'  # Ensure 'Unlabeled' appears at the bottom
        ),
        title=dict(
            text='',
            x=0.35,
            y=0.95,
            xanchor='center',
            yanchor='top',
            font=dict(size=20)
        ),
        xaxis=dict(
            title='TSNE1',
            title_font=dict(size=38),
            tickfont=dict(size=1)
        ),
        yaxis=dict(
            title='TSNE2',
            title_font=dict(size=38),
            tickfont=dict(size=1)
        ),
        margin=dict(l=20, r=100, t=50, b=20),
        plot_bgcolor='#fdfbf9'
    )
    
    fig.show()
    fig.write_image(
        "merged_topic_distribution.png",
        engine="kaleido",
        scale=4.0,
        width=2000,
        height=800
    )
    
    # Show topic distribution for labeled models only
    print("\nTopic group distribution (labeled data only):")
    for model in model_names:
        model_data = combined_viz[combined_viz['Model'] == model]
        topic_dist = model_data['Topic_Group'].value_counts()
        print(f"\n{model} topic distribution:")
        print(topic_dist)
    
    return combined_viz

model_names = ['deepseek-v3']
# model_names = ['sonnet']
# model_names = ['gpt-4o']
analyze_model_topics_with_unlabeled(logical_and_df, model_names, all_questions)

In [28]:
def analyze_model_topics_umap_with_unlabeled(
    logical_and_df: pd.DataFrame,
    model_names: list[str],
    all_questions: list[str]
) -> pd.DataFrame:
    """
    Analyze and visualize topic distribution between models (with known topics)
    plus additional unlabeled questions using MPNet embeddings and UMAP.
    The unlabeled questions are plotted in gray with alpha=0.5,
    but are excluded from the legend.
    """
    import pandas as pd
    import plotly.express as px
    import numpy as np
    from sentence_transformers import SentenceTransformer
    from umap import UMAP

    # Filter for specified models
    labeled_df = logical_and_df[logical_and_df['generator_model'].isin(model_names)].copy()

    # Build a DataFrame for the unlabeled questions
    unlabeled_df = pd.DataFrame({
        'question_body': all_questions,
        'generator_model': ['Unlabeled'] * len(all_questions),
        'topic': [np.nan] * len(all_questions)
    })

    # Combine labeled and unlabeled for a single embedding pass
    combined_df = pd.concat([labeled_df, unlabeled_df], ignore_index=True)

    # Initialize MPNet model
    mpnet = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

    # Get embeddings for combined questions
    old_texts = combined_df['question_body'].tolist()
    embeddings = mpnet.encode(old_texts, show_progress_bar=True)
    texts = []
    for text in old_texts:
        formatted = ""
        for i in range(0, len(text), 80):
            chunk = text[i:i+80]
            formatted += chunk
            if i + 80 < len(text):
                formatted += "<br>"
        texts.append(formatted)

    # UMAP dimensionality reduction
    umap_model = UMAP(n_components=2, n_neighbors=15, random_state=42)
    umap_embeddings = umap_model.fit_transform(embeddings)

    # Define topic group mapping
    topic_to_group = {
        # Energy & Environment
        'energy storage': 'Energy & Environment',
        'economics and energy': 'Energy & Environment',
        'energy': 'Energy & Environment',
        'energy and environment': 'Energy & Environment',
        'energy policy and sustainability': 'Energy & Environment',

        # Economics
        'economics': 'Economics',
        'economics and finance': 'Economics',

        # Politics
        'geopolitics': 'Politics',
        'economics and education': 'Politics',
        'geopolitics and national security': 'Politics',
        'labor market and technology': 'Politics',

        # Computing
        'technology and regulation': 'AI & Computer Science',
        'materials science and quantum computing': 'AI & Computer Science',
        'cryptography': 'AI & Computer Science',
        'ai and computing': 'AI & Computer Science',
        'artificial general intelligence': 'AI & Computer Science',
        'robotics': 'AI & Computer Science',
        'ai and robotics': 'AI & Computer Science',
        'cybersecurity': 'AI & Computer Science',

        # Social Media
        'social media and politics': 'Media',
        'social media': 'Media',
        'social media and regulation': 'Media',
        'social media and mental health': 'Media',
        'election misinformation': 'Media',
        'social media regulation': 'Media',
        'marketing and e-commerce': 'Media',
        'elections and misinformation': 'Media',

        # Sports
        'sports': 'Sports',
        'sports and olympics': 'Sports'
    }

    def get_topic_group(topic: str) -> str:
        if pd.isna(topic):
            return 'Static Question'
        return topic_to_group.get(topic.lower(), 'Other')

    # Construct visualization DataFrame
    viz_df = pd.DataFrame({
        'UMAP1': umap_embeddings[:, 0],
        'UMAP2': umap_embeddings[:, 1],
        'Model': combined_df['generator_model'].values,
        'Question': texts,
        'Topic_Group': combined_df['topic'].apply(get_topic_group)
    })

    # Define color scheme, including unlabeled as semi-transparent gray
    color_map = {
        'Energy & Environment': '#2ecc71',  # Green
        'Economics': '#e74c3c',  # Red
        'Politics': '#3498db',  # Blue
        'AI & Computer Science': '#9b59b6',  # Purple
        'Other': '#f39c12',  # Orange
        'Static Question': 'rgba(128, 128, 128, 0.2)'  # Light gray with alpha 0.2
    }

    # Reorder Topic_Group categories to put Static Question last
    topic_order = ['Energy & Environment', 'Economics', 'Politics', 'AI & Computer Science', 'Other', 'Static Question']
    viz_df['Topic_Group'] = pd.Categorical(viz_df['Topic_Group'], categories=topic_order, ordered=True)

    # Create interactive plot
    fig = px.scatter(
        viz_df,
        x='UMAP1',
        y='UMAP2',
        color='Topic_Group',
        color_discrete_map=color_map,
        category_orders={'Topic_Group': topic_order},
        hover_data={
            'UMAP1': False,
            'UMAP2': False,
            'Model': True,
            'Topic_Group': True,
            'Question': True
        },
        title='Topic Distribution (UMAP with Unlabeled)',
        height=800,
        width=2000
    )

    fig.update_traces(marker=dict(size=15))

    # Hide "Unlabeled" in the legend
    for trace in fig.data:
        if trace.name == 'Static Question':
            trace.name = 'Static Questions'
            # trace.showlegend = False

    # General layout updates
    fig.update_layout(
        template='plotly_white',
        legend=dict(
            title=dict(
                text='Question Topics',
                font=dict(size=32)
            ),
            yanchor="top",
            y=1.0,
            xanchor="left",
            x=0.02,
            bgcolor='#fdfbf9',
            font=dict(size=29),
            bordercolor="Black",
            borderwidth=0
        ),
        title=dict(
            text='',
            x=0.35,
            y=0.95,
            xanchor='center',
            yanchor='top',
            font=dict(size=20)
        ),
        xaxis=dict(
            title='UMAP1',
            title_font=dict(size=38),
            tickfont=dict(size=1)
        ),
        yaxis=dict(
            title='UMAP2',
            title_font=dict(size=38),
            tickfont=dict(size=1)
        ),
        margin=dict(l=20, r=100, t=50, b=20),
        plot_bgcolor='#fdfbf9'
    )

    fig.show()

    fig.write_image(
        "merged_topic_distribution_umap.png",
        engine="kaleido",
        scale=4.0,
        width=2000,
        height=800
    )

    # Show topic distribution only for labeled models
    print("\nTopic group distribution (labeled data only):")
    for model in model_names:
        model_data = viz_df[viz_df['Model'] == model]
        topic_dist = model_data['Topic_Group'].value_counts()
        print(f"\n{model} topic distribution:")
        print(topic_dist)

    return viz_df

In [ ]:
# Run the analysis
# model_names = ['gpt-4o', 'o1_mini', 'deepseek-v3', 'llama_70b', 'sonnet', 'llama_8b', 'gpt-4o-mini']
model_names = ['deepseek-v3']
# all_questions = Ps + Qs + PQs
# all_questions = PQs
viz_df = analyze_model_topics_umap_with_unlabeled(logical_and_df, model_names, all_questions)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Read the CSV file
# df = pd.read_csv('deepseek-v3/deepseek_chat_20250127_212346.csv')
df = pd.read_csv('deepseek-v3/deepseek_chat_20250127_124630.csv')

# Filter for only P_and_Q questions and drop duplicates
df_pq = df[df['question_type'] == 'P_and_Q'].drop_duplicates(subset=['question_triple_id'])

# Create a figure with subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Box plot to compare distributions
sns.boxplot(data=df_pq, x='iteration', y='consistency_score', ax=ax1)
ax1.set_title('P_and_Q Consistency Scores by Iteration')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Consistency Score')
ax1.grid(True, alpha=0.3)

# Plot 2: Violin plot to show distribution shapes
sns.violinplot(data=df_pq, x='iteration', y='consistency_score', ax=ax2)
ax2.set_title('Distribution of P_and_Q Consistency Scores by Iteration')
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Consistency Score')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary statistics by iteration
print("\nP_and_Q Summary Statistics by Iteration:")
print(df_pq.groupby('iteration')['consistency_score'].describe())

# Create a line plot showing the trend of mean scores
plt.figure(figsize=(10, 6))
mean_scores = df_pq.groupby('iteration')['consistency_score'].mean()
std_scores = df_pq.groupby('iteration')['consistency_score'].std()

plt.plot(mean_scores.index, mean_scores.values, marker='o', linewidth=2, label='Mean Score')
plt.fill_between(mean_scores.index, 
                 mean_scores.values - std_scores.values,
                 mean_scores.values + std_scores.values,
                 alpha=0.2)

plt.title('Mean P_and_Q Consistency Score Evolution Over Iterations')
plt.xlabel('Iteration')
plt.ylabel('Mean Consistency Score')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
df = pd.read_csv('deepseek-v3/deepseek_chat_20250127_124630.csv')

# Filter for only P_and_Q questions and drop duplicates
df_pq = df[df['question_type'] == 'P_and_Q'].drop_duplicates(subset=['question_triple_id'])


# Create a figure with subplots - now 2x2 grid
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(20, 12))

# Plot 1: Box plot (existing)
sns.boxplot(data=df_pq, x='iteration', y='consistency_score', ax=ax1)
ax1.set_title('P_and_Q Consistency Scores by Iteration')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Consistency Score')
ax1.grid(True, alpha=0.3)

# Plot 2: Violin plot (existing)
sns.violinplot(data=df_pq, x='iteration', y='consistency_score', ax=ax2)
ax2.set_title('Distribution of P_and_Q Consistency Scores by Iteration')
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Consistency Score')
ax2.grid(True, alpha=0.3)

# Plot 3: Line plot with mean and std (existing, moved to subplot)
mean_scores = df_pq.groupby('iteration')['consistency_score'].mean()
std_scores = df_pq.groupby('iteration')['consistency_score'].std()

ax3.plot(mean_scores.index, mean_scores.values, marker='o', linewidth=2, label='Mean Score')
ax3.fill_between(mean_scores.index, 
                mean_scores.values - std_scores.values,
                mean_scores.values + std_scores.values,
                alpha=0.2)
ax3.set_title('Mean P_and_Q Consistency Score Evolution')
ax3.set_xlabel('Iteration')
ax3.set_ylabel('Mean Consistency Score')
ax3.grid(True, alpha=0.3)
ax3.legend()

# Plot 4: Top 30 questions per iteration
def get_top_n_per_iteration(df, n=30):
    return df.groupby('iteration').apply(
        lambda x: x.nlargest(n, 'consistency_score')['consistency_score'].mean()
    )

top_20_means = get_top_n_per_iteration(df_pq, 20)
top_20_std = df_pq.groupby('iteration').apply(
    lambda x: x.nlargest(20, 'consistency_score')['consistency_score'].std()
)

ax4.plot(top_20_means.index, top_20_means.values, marker='o', linewidth=2, 
         label='Mean of Top 20', color='green')
ax4.fill_between(top_20_means.index,
                top_20_means.values - top_20_std.values,
                top_20_means.values + top_20_std.values,
                alpha=0.2, color='green')
ax4.set_title('Top 20 Questions Score Evolution')
ax4.set_xlabel('Iteration')
ax4.set_ylabel('Mean Score of Top 30')
ax4.grid(True, alpha=0.3)
ax4.legend()

plt.tight_layout()
plt.show()

# Print summary statistics
print("\nP_and_Q Summary Statistics by Iteration:")
print(df_pq.groupby('iteration')['consistency_score'].describe())
print("\nTop 20 Questions Summary by Iteration:")
print(top_20_means)


In [ ]:

# df = pd.read_csv('sonnet/anthropic_claude_3.5_sonnet_20250127_052908.csv')
# df = pd.read_csv('llama_70b/meta_llama_Meta_Llama_3.1_70B_Instruct_Turbo_20250126_232333.csv')
df = pd.read_csv('llama_70b/meta_llama_Meta_Llama_3.1_70B_Instruct_Turbo_20250126_212255.csv')
# df = pd.read_csv('o1-mini/o1_mini_20250127_042817.csv')
# df = pd.read_csv('o1-mini/o1_mini_20250127_045330.csv')

# Filter for only P_and_Q questions and drop duplicates
df_pq = df[df['question_type'] == 'P_and_Q'].drop_duplicates(subset=['question_triple_id'])

# Create a figure with subplots - now 2x2 grid
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(20, 12))

# Plot 1: Box plot (existing)
sns.boxplot(data=df_pq, x='iteration', y='consistency_score', ax=ax1)
ax1.set_title('P_and_Q Consistency Scores by Iteration')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Consistency Score')
ax1.grid(True, alpha=0.3)

# Plot 2: Violin plot (existing)
sns.violinplot(data=df_pq, x='iteration', y='consistency_score', ax=ax2)
ax2.set_title('Distribution of P_and_Q Consistency Scores by Iteration')
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Consistency Score')
ax2.grid(True, alpha=0.3)

# Plot 3: Line plot with mean and std (existing, moved to subplot)
mean_scores = df_pq.groupby('iteration')['consistency_score'].mean()
std_scores = df_pq.groupby('iteration')['consistency_score'].std()

ax3.plot(mean_scores.index, mean_scores.values, marker='o', linewidth=2, label='Mean Score')
ax3.fill_between(mean_scores.index, 
                mean_scores.values - std_scores.values,
                mean_scores.values + std_scores.values,
                alpha=0.2)
ax3.set_title('Mean P_and_Q Consistency Score Evolution')
ax3.set_xlabel('Iteration')
ax3.set_ylabel('Mean Consistency Score')
ax3.grid(True, alpha=0.3)
ax3.legend()

# Plot 4: Top 30 questions per iteration
def get_top_n_per_iteration(df, n=30):
    return df.groupby('iteration').apply(
        lambda x: x.nlargest(n, 'consistency_score')['consistency_score'].mean()
    )

top_20_means = get_top_n_per_iteration(df_pq, 20)
top_20_std = df_pq.groupby('iteration').apply(
    lambda x: x.nlargest(20, 'consistency_score')['consistency_score'].std()
)

ax4.plot(top_20_means.index, top_20_means.values, marker='o', linewidth=2, 
         label='Mean of Top 20', color='green')
ax4.fill_between(top_20_means.index,
                top_20_means.values - top_20_std.values,
                top_20_means.values + top_20_std.values,
                alpha=0.2, color='green')
ax4.set_title('Top 20 Questions Score Evolution')
ax4.set_xlabel('Iteration')
ax4.set_ylabel('Mean Score of Top 30')
ax4.grid(True, alpha=0.3)
ax4.legend()

plt.tight_layout()
plt.show()

# Print summary statistics
print("\nP_and_Q Summary Statistics by Iteration:")
print(df_pq.groupby('iteration')['consistency_score'].describe())
print("\nTop 20 Questions Summary by Iteration:")
print(top_20_means)


In [ ]:
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.manifold import TSNE

def plot_tsne_topic_trajectory():
    """
    Load:
      1) Base dataset (prithvi3/consistency_250_test) => iteration=0, colored light gray.
      2) llama_70b/meta_llama_Meta_Llama_3.1_70B_Instruct_Turbo_20250126_212255.csv => iteration>0, 
         colored by topic group with increasing opacity per iteration.
    Use TSNE to embed, then plot with a legend keyed by topic group, 
    showing formatted text per point in the hover.
    """

    # 1) Load the base dataset from Hugging Face
    ds_base = load_dataset("prithvi3/consistency_250_test")
    base_test = ds_base["test"]
    base_questions = [
        {
            "question_triple_id": idx,
            "question_body": item["P_and_Q_body"],
            "iteration": 0,
            "topic": None,  # no explicit topic from the base
        }
        for idx, item in enumerate(base_test)
    ]
    base_df = pd.DataFrame(base_questions)

    # 2) Load the llama_70b CSV
    llama_df = pd.read_csv(
        "llama_70b/meta_llama_Meta_Llama_3.1_70B_Instruct_Turbo_20250126_212255.csv"
    )
    # Keep only P_and_Q type
    llama_df = llama_df[llama_df["question_type"] == "P_and_Q"].copy()

    # 3) Combine base + llama_70b
    combined = pd.concat([base_df, llama_df], ignore_index=True)

    # Drop rows without question text
    combined.dropna(subset=["question_body"], inplace=True)

    # -------------------------
    # Topic grouping as in your snippet
    # -------------------------
    topic_to_group = {
        # Energy & Environment
        'energy storage': 'Energy & Environment',
        'economics and energy': 'Energy & Environment',
        'energy': 'Energy & Environment',
        'energy and environment': 'Energy & Environment',
        'energy policy and sustainability': 'Energy & Environment',
        
        # Economics
        'economics': 'Economics',
        'economics and finance': 'Economics',
        
        # Politics
        'geopolitics': 'Politics',
        'economics and education': 'Politics',
        'geopolitics and national security': 'Politics',
        'labor market and technology': 'Politics',
        
        # Computing
        'technology and regulation': 'AI & Computer Science',
        'materials science and quantum computing': 'AI & Computer Science',
        'cryptography': 'AI & Computer Science',
        'ai and computing': 'AI & Computer Science',
        'artificial general intelligence': 'AI & Computer Science',
        'robotics': 'AI & Computer Science',
        'ai and robotics': 'AI & Computer Science',
        'cybersecurity': 'AI & Computer Science',
        
        # Social Media
        'social media and politics': 'Media',
        'social media': 'Media',
        'social media and regulation': 'Media',
        'social media and mental health': 'Media',
        'election misinformation': 'Media',
        'social media regulation': 'Media',
        'marketing and e-commerce': 'Media',
        'elections and misinformation': 'Media',
        
        # Sports
        'sports': 'Sports',
        'sports and olympics': 'Sports'
    }

    def get_topic_group(topic: str) -> str:
        if pd.isna(topic) or not isinstance(topic, str):
            return 'Unlabeled'
        return topic_to_group.get(topic.lower(), 'Other')

    # Create a new column for the mapped topic group
    combined["Topic_Group"] = combined["topic"].apply(get_topic_group)

    # -------------------------
    # Embed with MPNet
    # -------------------------
    mpnet = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
    raw_texts = combined["question_body"].tolist()

    # (Optional) Format text for hover readability
    formatted_texts = []
    for text in raw_texts:
        chunks = []
        for i in range(0, len(text), 80):
            chunks.append(text[i:i+80])
        formatted_texts.append("<br>".join(chunks))

    embeddings = mpnet.encode(raw_texts, show_progress_bar=True)

    # Run TSNE
    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    coords = tsne.fit_transform(embeddings)

    combined["tsne_x"] = coords[:, 0]
    combined["tsne_y"] = coords[:, 1]
    combined["formatted_text"] = formatted_texts

    # -------------------------
    # Define color map for each topic group
    # -------------------------
    color_map = {
        'Energy & Environment': '#2ecc71',
        'Economics': '#e74c3c',
        'Politics': '#3498db',
        'AI & Computer Science': '#9b59b6',
        'Media': '#f1c40f',
        'Sports': '#e67e22',
        'Other': '#95a5a6',
        'Unlabeled': 'rgba(128, 128, 128, 0.2)',
        # We'll also define 'Static Questions' as the base iteration
        'Static Questions': '#bdc3c7',  # light gray
    }

    # Helper: hex -> (r,g,b)
    def hex_to_rgb(hex_color: str) -> tuple[int, int, int]:
        hex_color = hex_color.lstrip('#')
        return tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))

    # Helper to add alpha
    def color_with_alpha(hx: str, alpha: float) -> str:
        if hx.startswith("rgba(") or hx.startswith("rgb("):
            # Already has RGBA or RGB
            return hx
        rgb = hex_to_rgb(hx)
        return f"rgba({rgb[0]},{rgb[1]},{rgb[2]},{alpha:.2f})"

    # We'll transform the data so that iteration=0 => "Static Questions" as topic group to appear in legend
    def map_iteration_to_color(row: pd.Series) -> str:
        iteration = row["iteration"]
        group = row["Topic_Group"]
        if iteration == 0:
            # Base => light gray (fully opaque)
            return color_map["Static Questions"]
        # For iteration>0 => color_map for that group, but with alpha scaling
        alpha = min(1.0, 0.2 + 0.15 * iteration)  # e.g. iteration=1 => 0.35, iteration=2 => 0.50, ...
        base_color = color_map.get(group, "#95a5a6")  # default to 'Other' gray if missing
        return color_with_alpha(base_color, alpha)

    # We group by "Topic_Group" so we can have a single trace per group
    # That way each group has one legend entry
    fig = go.Figure()

    shown_legend_for = set()  # keep track so each group is shown once

    for topic_group, group_df in combined.groupby("Topic_Group"):
        # We'll compute a color per point
        point_colors = group_df.apply(map_iteration_to_color, axis=1)
        # Build hover text
        hover_texts = [
            f"Topic Group: {topic_group}<br>"
            f"Iteration: {row.iteration}<br>"
            f"{row.formatted_text}"
            for _, row in group_df.iterrows()
        ]

        # Decide how to label the legend
        if topic_group == "Unlabeled" or topic_group == "Other":
            # If you want them hidden or renamed, do so here
            legend_name = topic_group
        elif topic_group == "Static Questions":
            legend_name = "Static Questions"
        else:
            legend_name = topic_group

        showlegend = topic_group not in shown_legend_for
        fig.add_trace(
            go.Scatter(
                x=group_df["tsne_x"],
                y=group_df["tsne_y"],
                mode="markers",
                text=hover_texts,
                hoverinfo="text",
                marker=dict(color=point_colors, size=12),
                name=legend_name,
                showlegend=showlegend,
                legendgroup=topic_group
            )
        )
        shown_legend_for.add(topic_group)

    # -------------------------
    # Layout / legend configuration
    # -------------------------
    fig.update_layout(
        title="Question Trajectory by Topic Group",
        xaxis_title="TSNE 1",
        yaxis_title="TSNE 2",
        template='plotly_white',
        width=2000,
        height=800,
        legend=dict(
            title=dict(text="Question Topics", font=dict(size=18)),
            yanchor="top",
            y=1.0,
            xanchor="right",
            x=0.98,
            bgcolor='#fdfbf9',
            font=dict(size=14)
        )
    )

    fig.show()

    # Optionally save the figure
    fig.write_image(
        "colored_tsne_trajectory.png",
        engine="kaleido",
        scale=3.0,
        width=2000,
        height=800
    )

    return combined

df_plotted = plot_tsne_topic_trajectory()
print("Plotted DataFrame shape:", df_plotted.shape)

In [ ]:
def plot_tsne_topic_trajectory():
    """
    Load:
      1) Base dataset (prithvi3/consistency_250_test) => iteration=0 (label "Base Dataset" in legend, light gray color).
      2) llama_70b CSV => final iteration only, colored by topic group.

    Then embed questions using TSNE and plot:
      - iteration=0 => "Base Dataset" (light gray),
      - iteration=final => color by the mapped topic group.
    """

    import os
    import pandas as pd
    import numpy as np
    import plotly.graph_objects as go
    from datasets import load_dataset
    from sentence_transformers import SentenceTransformer
    from sklearn.manifold import TSNE

    # 1) Load the base dataset
    ds_base = load_dataset("prithvi3/consistency_250_test")
    base_test = ds_base["test"]
    base_questions = [
        {
            "question_triple_id": idx,
            "question_body": item["P_and_Q_body"],
            "iteration": -1,
            "topic": None,  # no explicit topic from the base
        }
        for idx, item in enumerate(base_test)
    ]
    base_df = pd.DataFrame(base_questions)

    # 2) Load the llama_70b CSV
    llama_df = pd.read_csv(
        "llama_70b/meta_llama_Meta_Llama_3.1_70B_Instruct_Turbo_20250126_212255.csv"
    )
    llama_df = llama_df[llama_df["question_type"] == "P_and_Q"].copy()

    # 3) Combine base + llama_70b
    combined = pd.concat([base_df, llama_df], ignore_index=True).dropna(subset=["question_body"])

    # -------------------------
    # Topic grouping
    # -------------------------
    topic_to_group = {
        # Energy & Environment
        'energy storage': 'Energy & Environment',
        'economics and energy': 'Energy & Environment',
        'energy': 'Energy & Environment',
        'energy and environment': 'Energy & Environment',
        'energy policy and sustainability': 'Energy & Environment',
        'environment and sustainability': 'Energy & Environment',
        'energy and environment': 'Energy & Environment',
        'environment and business': 'Energy & Environment',
        'environment and technology': 'Energy & Environment',
        'environmental impact and technology': 'Energy & Environment',
        'environmental sustainability and tech industry': 'Energy & Environment',
        'environment and operations': 'Energy & Environment',

        # Economics
        'economics': 'Economics',
        'economics and finance': 'Economics',
        'economics and technology': 'Economics',
        'economics and education': 'Economics',

        # Politics
        'geopolitics': 'Politics',
        'geopolitics and national security': 'Politics',
        'politics and social media': 'Politics',

        # AI & Computer Science
        'technology and regulation': 'AI & Computer Science',
        'materials science and quantum computing': 'AI & Computer Science',
        'cryptography': 'AI & Computer Science',
        'ai and computing': 'AI & Computer Science',
        'artificial general intelligence': 'AI & Computer Science',
        'robotics': 'AI & Computer Science',
        'ai and robotics': 'AI & Computer Science',
        'cybersecurity': 'AI & Computer Science',
        'artificial intelligence and quantum computing': 'AI & Computer Science',
        'quantum computing and ai': 'AI & Computer Science',

        # Media
        'social media and politics': 'Media',
        'social media': 'Media',
        'social media and regulation': 'Media',
        'social media and mental health': 'Media',
        'election misinformation': 'Media',
        'social media regulation': 'Media',
        'marketing and e-commerce': 'Media',
        'elections and misinformation': 'Media',
        'marketing and social media': 'Media',
        'social media and electric vehicles': 'Media',
        'social media and e-commerce': 'Media',
        'social media and sales': 'Media',

        # Transportation
        'autonomous vehicles and transportation': 'Transportation',
        'transportation and technology': 'Transportation',
        'transportation and infrastructure': 'Transportation',
        'transportation and autonomous vehicles': 'Transportation',
        'autonomous vehicles': 'Transportation',
        'autonomous vehicles and traffic accidents': 'Transportation',

        # Agriculture & Biotechnology
        'agriculture and biotechnology': 'Agriculture & Biotechnology',
        'agriculture and genetic engineering': 'Agriculture & Biotechnology',
        'genetic engineering': 'Agriculture & Biotechnology',

        # Sports
        'sports': 'Sports',
        'sports and olympics': 'Sports',

        # Global Events
        'global events and supply chains': 'Global Events',
    }

    def get_topic_group(topic: str) -> str:
        if pd.isna(topic) or not isinstance(topic, str):
            return 'Unlabeled'
        return topic_to_group.get(topic.lower(), 'Other')

    combined["Topic_Group"] = combined["topic"].apply(get_topic_group)

    # -------------------------
    # Embed with MPNet
    # -------------------------
    mpnet = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
    raw_texts = combined["question_body"].tolist()

    # Format text for nicer hover
    formatted_texts = []
    for text in raw_texts:
        chunks = []
        for i in range(0, len(text), 80):
            chunks.append(text[i:i+80])
        formatted_texts.append("<br>".join(chunks))

    embeddings = mpnet.encode(raw_texts, show_progress_bar=True)

    # Run TSNE
    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    coords = tsne.fit_transform(embeddings)

    combined["tsne_x"] = coords[:, 0]
    combined["tsne_y"] = coords[:, 1]
    combined["formatted_text"] = formatted_texts

    # -------------------------
    # Only keep iteration=0 and iteration=final
    # -------------------------
    # final_iter = combined["iteration"].max()
    # combined = combined[combined["iteration"].isin([0, final_iter])]

    # -------------------------
    # Prepare legend labels and colors
    # -------------------------
    def get_legend_label(iteration: int, topic_group: str) -> str:
        return "Base Dataset" if iteration == 0 else topic_group

    # Light gray for base dataset
    BASE_COLOR = "#bdc3c7"

    color_map = {
        "Energy & Environment": "#2ecc71",
        "Economics": "#e74c3c",
        "Politics": "#3498db",
        "AI & Computer Science": "#9b59b6",
        "Media": "#f1c40f",
        "Sports": "#e67e22",
        "Other": "#95a5a6",
        "Unlabeled": "rgba(128, 128, 128, 0.2)",
    }

    def get_color(iteration: int, topic_group: str) -> str:
        return BASE_COLOR if iteration == 0 else color_map.get(topic_group, "#95a5a6")

    # Compute the legend label for each row
    combined["Legend_Label"] = combined.apply(
        lambda r: get_legend_label(r["iteration"], r["Topic_Group"]),
        axis=1
    )

    # -------------------------
    # Plot
    # -------------------------
    fig = go.Figure()
    shown_legend = set()

    for legend_label, group_df in combined.groupby("Legend_Label"):
        point_colors = [
            get_color(row["iteration"], row["Topic_Group"])
            for _, row in group_df.iterrows()
        ]
        hover_texts = [
            f"Iteration: {row.iteration}<br>"
            f"Topic Group: {row.Topic_Group}<br>"
            f"{row.formatted_text}"
            for _, row in group_df.iterrows()
        ]

        showlegend = legend_label not in shown_legend
        fig.add_trace(
            go.Scatter(
                x=group_df["tsne_x"],
                y=group_df["tsne_y"],
                mode="markers",
                text=hover_texts,
                hoverinfo="text",
                marker=dict(color=point_colors, size=12),
                name=legend_label,
                showlegend=showlegend,
                legendgroup=legend_label
            )
        )
        shown_legend.add(legend_label)

    # -------------------------
    # Layout / legend
    # -------------------------
    fig.update_layout(
        title="TSNE: Base (iteration=0) vs. Final Iteration",
        xaxis_title="TSNE 1",
        yaxis_title="TSNE 2",
        template='plotly_white',
        width=2000,
        height=800,
        legend=dict(
            title=dict(text="Topic Groups", font=dict(size=18)),
            yanchor="top",
            y=1.0,
            xanchor="right",
            x=0.98,
            bgcolor='#fdfbf9',
            font=dict(size=14)
        )
    )

    fig.show()

    fig.write_image(
        "colored_tsne_trajectory_base_vs_final.png",
        engine="kaleido",
        scale=3.0,
        width=2000,
        height=800
    )

    return combined

df_combined = plot_tsne_topic_trajectory()

In [10]:
ds_base = load_dataset("prithvi3/consistency_250_test")
base_test = ds_base["test"]
base_questions = [
    {
        "question_triple_id": idx,
        "question_body": item["P_and_Q_body"],
        "iteration": -1,
        "topic": None,  # no explicit topic from the base
    }
    for idx, item in enumerate(base_test)
]
base_df = pd.DataFrame(base_questions)

# 2) Load the llama_70b CSV
llama_df = pd.read_csv(
    "llama_70b/meta_llama_Meta_Llama_3.1_70B_Instruct_Turbo_20250126_212255.csv"
)
llama_df = llama_df[llama_df["question_type"] == "P_and_Q"].copy()

# 3) Combine base + llama_70b
combined = pd.concat([base_df, llama_df], ignore_index=True).dropna(subset=["question_body"])


In [ ]:
combined.tail()

In [ ]:
list(set(combined["topic"]))

In [ ]:
combined.head()

In [ ]:
    topic_to_group = {
        # Energy & Environment
        'energy storage': 'Energy & Environment',
        'economics and energy': 'Energy & Environment',
        'energy': 'Energy & Environment',
        'energy and environment': 'Energy & Environment',
        'energy policy and sustainability': 'Energy & Environment',
        'environment and sustainability': 'Energy & Environment',
        'energy and environment': 'Energy & Environment',
        'environment and business': 'Energy & Environment',
        'environment and technology': 'Energy & Environment',
        'environmental impact and technology': 'Energy & Environment',
        'environmental sustainability and tech industry': 'Energy & Environment',
        'environment and operations': 'Energy & Environment',

        # Economics
        'economics': 'Economics',
        'economics and finance': 'Economics',
        'economics and technology': 'Economics',
        'economics and education': 'Economics',

        # Politics
        'geopolitics': 'Politics',
        'geopolitics and national security': 'Politics',
        'politics and social media': 'Politics',

        # AI & Computer Science
        'technology and regulation': 'AI & Computer Science',
        'materials science and quantum computing': 'AI & Computer Science',
        'cryptography': 'AI & Computer Science',
        'ai and computing': 'AI & Computer Science',
        'artificial general intelligence': 'AI & Computer Science',
        'robotics': 'AI & Computer Science',
        'ai and robotics': 'AI & Computer Science',
        'cybersecurity': 'AI & Computer Science',
        'artificial intelligence and quantum computing': 'AI & Computer Science',
        'quantum computing and ai': 'AI & Computer Science',

        # Media
        'social media and politics': 'Media',
        'social media': 'Media',
        'social media and regulation': 'Media',
        'social media and mental health': 'Media',
        'election misinformation': 'Media',
        'social media regulation': 'Media',
        'marketing and e-commerce': 'Media',
        'elections and misinformation': 'Media',
        'marketing and social media': 'Media',
        'social media and electric vehicles': 'Media',
        'social media and e-commerce': 'Media',
        'social media and sales': 'Media',

        # Transportation
        'autonomous vehicles and transportation': 'Transportation',
        'transportation and technology': 'Transportation',
        'transportation and infrastructure': 'Transportation',
        'transportation and autonomous vehicles': 'Transportation',
        'autonomous vehicles': 'Transportation',
        'autonomous vehicles and traffic accidents': 'Transportation',

        # Agriculture & Biotechnology
        'agriculture and biotechnology': 'Agriculture & Biotechnology',
        'agriculture and genetic engineering': 'Agriculture & Biotechnology',
        'genetic engineering': 'Agriculture & Biotechnology',

        # Sports
        'sports': 'Sports',
        'sports and olympics': 'Sports',

        # Global Events
        'global events and supply chains': 'Global Events',
    }

    def get_topic_group(topic: str) -> str:
        if pd.isna(topic) or not isinstance(topic, str):
            return 'Unlabeled'
        return topic_to_group.get(topic.lower(), 'Other')

    combined["Topic_Group"] = combined["topic"].apply(get_topic_group)

In [ ]:
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.manifold import TSNE


def plot_tsne_topic_trajectory(optimization_path :str = "llama_70b/meta_llama_Meta_Llama_3.1_70B_Instruct_Turbo_20250126_212255.csv") -> pd.DataFrame:
    """
    Load:
      1) Base dataset (prithvi3/consistency_250_test) => iteration=-1 (label "Base Dataset" in legend, light gray color).
      2) llama_70b CSV => any iteration ≥ 0, colored by topic group with increasing opacity per iteration.

    Then embed questions using TSNE and plot:
      - iteration < 0 => "Base Dataset" (fully opaque light gray),
      - iteration ≥ 0 => color by the mapped topic group (including Other) with alpha based on iteration.
    """

    # 1) Load the base dataset
    ds_base = load_dataset("prithvi3/cond_100_test")
    base_test = ds_base["test"]
    base_questions = [
        {
            "question_triple_id": idx,
            "question_body": item["P_and_Q_body"],
            "iteration": -1,  # iteration < 0 for base dataset
            "topic": None,
        }
        for idx, item in enumerate(base_test)
    ]
    base_df = pd.DataFrame(base_questions)

    llama_df = pd.read_csv(
        optimization_path
    )
    llama_df = llama_df[llama_df["question_type"] == "P_and_Q"].copy()

    combined = pd.concat([base_df, llama_df], ignore_index=True).dropna(subset=["question_body"])

    # -------------------------
    # Topic grouping
    # -------------------------
    topic_to_group = {
        # Energy & Environment
        'energy storage': 'Energy & Environment',
        'economics and energy': 'Energy & Environment',
        'energy': 'Energy & Environment',
        'energy and environment': 'Energy & Environment',
        'energy policy and sustainability': 'Energy & Environment',
        'environment and sustainability': 'Energy & Environment',
        'environment and business': 'Energy & Environment',
        'environment and technology': 'Energy & Environment',
        'environmental impact and technology': 'Energy & Environment',
        'environmental sustainability and tech industry': 'Energy & Environment',
        'environment and operations': 'Energy & Environment',

        # Economics
        'economics': 'Economics',
        'economics and finance': 'Economics',
        'economics and technology': 'Economics',
        'economics and education': 'Economics',

        # Politics
        'geopolitics': 'Politics',
        'geopolitics and national security': 'Politics',
        'politics and social media': 'Politics',

        # AI & Computer Science
        'technology and regulation': 'AI & Computer Science',
        'materials science and quantum computing': 'AI & Computer Science',
        'cryptography': 'AI & Computer Science',
        'ai and computing': 'AI & Computer Science',
        'artificial general intelligence': 'AI & Computer Science',
        'robotics': 'AI & Computer Science',
        'ai and robotics': 'AI & Computer Science',
        'cybersecurity': 'AI & Computer Science',
        'artificial intelligence and quantum computing': 'AI & Computer Science',
        'quantum computing and ai': 'AI & Computer Science',

        # Media
        'social media and politics': 'Media',
        'social media': 'Media',
        'social media and regulation': 'Media',
        'social media and mental health': 'Media',
        'election misinformation': 'Media',
        'social media regulation': 'Media',
        'marketing and e-commerce': 'Media',
        'elections and misinformation': 'Media',
        'marketing and social media': 'Media',
        'social media and electric vehicles': 'Media',
        'social media and e-commerce': 'Media',
        'social media and sales': 'Media',

        # Transportation
        'autonomous vehicles and transportation': 'Transportation',
        'transportation and technology': 'Transportation',
        'transportation and infrastructure': 'Transportation',
        'transportation and autonomous vehicles': 'Transportation',
        'autonomous vehicles': 'Transportation',
        'autonomous vehicles and traffic accidents': 'Transportation',

        # Agriculture & Biotechnology
        'agriculture and biotechnology': 'Agriculture & Biotechnology',
        'agriculture and genetic engineering': 'Agriculture & Biotechnology',
        'genetic engineering': 'Agriculture & Biotechnology',

        # Sports
        'sports': 'Sports',
        'sports and olympics': 'Sports',

        # Global Events
        'global events and supply chains': 'Global Events',
    }

    def get_topic_group(topic: str) -> str:
        """
        If topic is NaN or not a string, or not in topic_to_group,
        map it to 'Other' (conflating both 'Unlabeled' and 'Other').
        """
        if pd.isna(topic) or not isinstance(topic, str):
            return 'Other'
        return topic_to_group.get(topic.lower(), 'Other')

    combined["Topic_Group"] = combined["topic"].apply(get_topic_group)

    # -------------------------
    # Embed with MPNet
    # -------------------------
    mpnet = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
    raw_texts = combined["question_body"].tolist()

    # Format text for nicer hover
    formatted_texts: list[str] = []
    for text in raw_texts:
        chunks: list[str] = []
        for i in range(0, len(text), 80):
            chunks.append(text[i : i + 80])
        formatted_texts.append("<br>".join(chunks))

    embeddings = mpnet.encode(raw_texts, show_progress_bar=True)

    # Run TSNE
    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    coords = tsne.fit_transform(embeddings)

    combined["tsne_x"] = coords[:, 0]
    combined["tsne_y"] = coords[:, 1]
    combined["formatted_text"] = formatted_texts

    # -------------------------
    # Prepare legend labels
    # -------------------------
    def get_legend_label(iteration: int, topic_group: str) -> str:
        # iteration < 0 => "Base Dataset"
        return "Base Dataset" if iteration < 0 else topic_group

    combined["Legend_Label"] = combined.apply(
        lambda r: get_legend_label(r["iteration"], r["Topic_Group"]),
        axis=1
    )

    #filter iterations to only include 0, 5, 10
    combined = combined[combined["iteration"].isin([-1, 0, 4, 5, 6, 10, 11, 12])]

    # -------------------------
    # Colors and alpha
    # -------------------------
    BASE_COLOR: str = "#E0E0E0"  # neutral light gray for base dataset
    color_map: dict[str, str] = {
        "Energy & Environment": "#4CAF50",      # Material Green
        "Economics": "#F44336",                 # Material Red 
        "Politics": "#2196F3",                  # Material Blue
        "AI & Computer Science": "#9C27B0",     # Material Purple
        "Media": "#FFC107",                     # Material Amber
        "Transportation": "#00BCD4",            # Material Cyan
        "Agriculture & Biotechnology": "#009688", # Material Teal
        "Global Events": "#607D8B",             # Material Blue Gray
        "Sports": "#FF9800",                    # Material Orange
        "Other": "#455A64",                     # Darker Blue Gray
    }

    def hex_to_rgb(hex_color: str) -> tuple[int, int, int]:
        hex_str = hex_color.lstrip('#')
        return tuple(int(hex_str[i : i + 2], 16) for i in range(0, len(hex_str), 2))

    def color_with_alpha(hex_color: str, alpha_val: float) -> str:
        rgb = hex_to_rgb(hex_color)
        return f"rgba({rgb[0]},{rgb[1]},{rgb[2]},{alpha_val:.2f})"

    def get_color(row: pd.Series) -> str:
        iteration: int = row["iteration"]
        topic_group: str = row["Topic_Group"]

        # Base dataset => fully opaque gray
        if iteration < 0:
            return BASE_COLOR

        # For iteration >= 0, define alpha to smoothly increase up to iteration 11
        # iteration=0 => alpha=0.20, iteration=11 => alpha=1.0
        max_iterations = 11
        alpha = 0.20 + (iteration / max_iterations) * 0.80
        alpha = min(1.0, alpha)

        base_hex = color_map.get(topic_group, "#2c3e50")  # fallback to "Other" color
        return color_with_alpha(base_hex, alpha)

    # -------------------------
    # Plot
    # -------------------------
    fig = go.Figure()
    shown_legend: set[str] = set()

    for legend_label, group_df in combined.groupby("Legend_Label"):
        point_colors: list[str] = [get_color(row) for _, row in group_df.iterrows()]
        hover_texts: list[str] = [
            f"Iteration: {row.iteration}<br>"
            f"Topic Group: {row.Topic_Group}<br>"
            f"{row.formatted_text}"
            for _, row in group_df.iterrows()
        ]

        showlegend: bool = legend_label not in shown_legend
        fig.add_trace(
            go.Scatter(
                x=group_df["tsne_x"],
                y=group_df["tsne_y"],
                mode="markers",
                text=hover_texts,
                hoverinfo="text",
                marker=dict(color=point_colors, size=16),
                name=legend_label,
                showlegend=showlegend,
                legendgroup=legend_label,
            )
        )
        shown_legend.add(legend_label)

    fig.update_layout(
        title="TSNE: Base vs. Iterations (Color Opacity by Iteration)",
        xaxis_title="TSNE 1",
        yaxis_title="TSNE 2",
        template='plotly_white',
        width=2000,
        height=800,
        legend=dict(
            title=dict(text="Topic Groups", font=dict(size=18)),
            yanchor="top",
            y=1.0,
            xanchor="right",
            x=0.98,
            bgcolor='#fdfbf9',
            font=dict(size=14),
        ),
    )

    fig.show()

    fig.write_image(
        "colored_tsne_trajectory_with_iteration_opacity.png",
        engine="kaleido",
        scale=3.0,
        width=2000,
        height=800
    )

    return combined

In [ ]:
combined = plot_tsne_topic_trajectory("/Users/davisbrown/adaptive_evals/tasks/cond_100/deepseek-v3/deepseek_chat_20250127_124630.csv")

In [79]:
topic_to_group = {
    # Energy & Environment
    'energy storage': 'Energy & Environ',
    'economics and energy': 'Energy & Environ',
    'energy': 'Energy & Environ',
    'energy and environment': 'Energy & Environ',
    'energy policy and sustainability': 'Energy & Environ',
    'environment and sustainability': 'Energy & Environ',
    'environment and business': 'Energy & Environ',
    'environment and technology': 'Energy & Environ',
    'environmental impact and technology': 'Energy & Environ',
    'environmental sustainability and tech industry': 'Energy & Environ',
    'environment and operations': 'Energy & Environ',
    'electric vehicle manufacturing': 'Energy & Environ',
    'climate change and food security': 'Energy & Environ',
    'renewable energy': 'Energy & Environ',
    'environmental factors and tech industry': 'Energy & Environ',

    # AI & Computer Science
    'technology and regulation': 'Tech & AI',
    'materials science and quantum computing': 'Tech & AI',
    'cryptography': 'Tech & AI',
    'ai and computing': 'Tech & AI',
    'artificial general intelligence': 'Tech & AI',
    'robotics': 'Tech & AI',
    'ai and robotics': 'Tech & AI',
    'cybersecurity': 'Tech & AI',
    'artificial intelligence and quantum computing': 'Tech & AI',
    'quantum computing and ai': 'Tech & AI',

    # Media
    'social media and politics': 'Media',
    'social media and us presidential election': 'Media',
    'social media': 'Media',
    'social media and regulation': 'Media',
    'social media and mental health': 'Media',
    'election misinformation': 'Media',
    'social media regulation': 'Media',
    'marketing and e-commerce': 'Media',
    'elections and misinformation': 'Media',
    'marketing and social media': 'Media',
    'social media and electric vehicles': 'Media',
    'social media and e-commerce': 'Media',
    'social media and sales': 'Media',

    # Transportation
    'autonomous vehicles and transportation': 'Manufacturing',
    'transportation and technology': 'Manufacturing',
    'transportation and infrastructure': 'Manufacturing',
    'transportation and autonomous vehicles': 'Manufacturing',
    'autonomous vehicles': 'Manufacturing',
    'autonomous vehicles and traffic accidents': 'Manufacturing',

    # Agriculture & Biotechnology
    'agriculture and biotechnology': 'Ag & Biotech',
    'agriculture and genetic engineering': 'Ag & Biotech',
    'genetic engineering': 'Ag & Biotech',

    # Sports
    'sports': 'Sports',
    'sports and olympics': 'Sports',

    # Global Events / Politics / Economics
    'global events and supply chains': 'Politics & Econ',
    'economics': 'Politics & Econ',
    'economics and finance': 'Politics & Econ',
    'economics and technology': 'Politics & Econ',
    'economics and education': 'Politics & Econ',
    'economic sanctions and international trade': 'Politics & Econ',
    'geopolitics': 'Politics & Econ',
    'geopolitics and national security': 'Politics & Econ',
    'politics and social media': 'Politics & Econ',
    'international trade and economic sanctions': 'Politics & Econ',

}

In [ ]:
import os
import pandas as pd
import numpy as np
from typing import List
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.manifold import TSNE
import plotly.express as px
import plotly.graph_objects as go

def analyze_model_topics(csv_path: str = "llama_70b/meta_llama_Meta_Llama_3.1_70B_Instruct_Turbo_20250126_212255.csv") -> pd.DataFrame:
    """
    Creates and combines base and llama_70b datasets internally, then analyzes and visualizes
    topic distribution between the base (iteration < 0) and optimization iterations (iteration >= 0)
    using MPNet embeddings and t-SNE. The plot is labeled with "Base", "Iteration 0", "Iteration 1", etc.
    
    Returns:
        A pandas DataFrame containing the TSNE embeddings and related information.
    """

    # -------------------------
    # 1) Base dataset
    # -------------------------
    ds_base = load_dataset("prithvi3/consistency_250_test")
    base_test = ds_base["test"]
    base_questions = []
    for idx, item in enumerate(base_test):
        base_questions.append({
            "question_body": item["P_and_Q_body"],
            "topic": None,
            "iteration": -1,         # signify base questions
            "generator_model": "Base" 
        })
    base_df = pd.DataFrame(base_questions)

    # -------------------------
    # 2) CSV
    # -------------------------
    model_df = pd.read_csv(csv_path)

    # Ensure these columns exist; if not, create defaults
    if "iteration" not in model_df.columns:
        model_df["iteration"] = 0
    if "generator_model" not in model_df.columns:
        model_df["generator_model"] = "llama_70b"

    # Keep only valid question bodies
    model_df = model_df.dropna(subset=["question_body"]).copy()

    # -------------------------
    # 3) Combine base + llama
    # -------------------------
    combined_df = pd.concat([base_df, model_df], ignore_index=True).dropna(subset=["question_body"])

    # If there are no rows matching the specified models, exit early
    if model_df.empty:
        print("No data found for the specified model names.")
        return pd.DataFrame()

    # -------------------------
    # Create a human-friendly iteration label
    # -------------------------
    def make_iteration_label(it: int) -> str:
        return "Base" if it < 0 else f"Iteration {it}"

    # -------------------------
    # Prepare data for t-SNE
    # -------------------------
    iteration_labels = model_df["iteration"].apply(make_iteration_label)
    texts = model_df["question_body"].tolist()

    # -------------------------
    # MPNet embeddings
    # -------------------------
    mpnet = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
    embeddings = mpnet.encode(texts, show_progress_bar=True)

    # -------------------------
    # t-SNE dimensionality reduction
    # -------------------------
    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    tsne_embeddings = tsne.fit_transform(embeddings)

    # -------------------------
    # Create visualization DataFrame
    # -------------------------
    viz_df = pd.DataFrame({
        "TSNE1": tsne_embeddings[:, 0],
        "TSNE2": tsne_embeddings[:, 1],
        "IterationLabel": iteration_labels.values,
        "iteration": model_df["iteration"].values,
        "Question": texts,
        "Question_Preview": [t[:100] + "..." for t in texts],
        "Topic": model_df["topic"].values
    })

    # -------------------------
    # Create interactive plot with gradient colors
    # -------------------------
    fig = px.scatter(
        viz_df,
        x="TSNE1",
        y="TSNE2",
        color="iteration",        # color by iteration number for gradient
        color_continuous_scale="Viridis",  # gradient from light to dark
        hover_data={
            "TSNE1": False,
            "TSNE2": False,
            "IterationLabel": True,
            "Topic": True,
            "Question_Preview": True
        },
        title="Base vs. Optimization Dataset Distribution (MPNet Embeddings)",
        labels={"Question_Preview": "Question"},
        height=800
    )

    # Optionally annotate topic centroids
    unique_topics = viz_df["Topic"].dropna().unique()
    for topic in unique_topics:
        topic_points = viz_df[viz_df["Topic"] == topic]
        # Skip if no points
        if topic_points.empty:
            continue

        x_mean = topic_points["TSNE1"].mean()
        y_mean = topic_points["TSNE2"].mean()

        fig.add_annotation(
            x=x_mean,
            y=y_mean,
            text=topic,
            showarrow=True,
            arrowhead=1,
            arrowsize=1,
            arrowwidth=2,
            arrowcolor="#636363",
            font=dict(size=18, color="#636363"),
            bgcolor="white",
            bordercolor="#c7c7c7",
            borderwidth=1,
            borderpad=4,
            opacity=0.8
        )

    # -------------------------
    # Update marker size and layout
    # -------------------------
    fig.update_traces(
        marker=dict(size=10),
        selector=dict(mode="markers")
    )
    fig.update_layout(
        legend=dict(
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=0.01
        ),
        plot_bgcolor="white",
        coloraxis_colorbar_title="Iteration"
    )

    # Show plot
    fig.show()

    # -------------------------
    # Print topic distribution by iteration label
    # -------------------------
    print("\nTopic distribution by iteration label:")
    for iteration_label in viz_df["IterationLabel"].unique():
        subset = viz_df[viz_df["IterationLabel"] == iteration_label]
        topic_dist = subset["Topic"].value_counts()
        print(f"\n{iteration_label} topic distribution:")
        print(topic_dist)

    return viz_df


# Example usage outside this function:
# model_names = ["llama_70b"]  # or your desired set of 'generator_model' values
viz_df = analyze_model_topics()

In [43]:
from typing import Optional
import os
import pandas as pd
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.manifold import TSNE
from sklearn.metrics import pairwise_distances
import plotly.express as px
import plotly.graph_objects as go

def analyze_model_topics(csv_path: str = "llama_70b/meta_llama_Meta_Llama_3.1_70B_Instruct_Turbo_20250126_212255.csv") -> pd.DataFrame:
    """
    Combines a base dataset (iteration < 0, plotted in very light grey) with a model CSV (iteration >= 0).
    We color the scatter plot by topic groups. We also compute a topic group per row
    (except base), then annotate each group with its medoid (rather than centroid).

    Returns:
        A pandas DataFrame containing TSNE embeddings and other columns.
    """

    # 1) Load the base dataset; mark iteration < 0
    # ds_base = load_dataset("prithvi3/consistency_250_test")
    ds_base = load_dataset("prithvi3/cond_100_test")
    base_test = ds_base["test"]
    base_questions = []
    for idx, item in enumerate(base_test):
        base_questions.append({
            "question_body": item["P_and_Q_body"],
            "topic": None,  # No explicit topic for base
            "iteration": -1,
            "generator_model": "Base"
        })
    base_df = pd.DataFrame(base_questions)

    # 2) Load CSV data
    model_df = pd.read_csv(csv_path).dropna(subset=["question_body"]).copy()
    if "iteration" not in model_df.columns:
        model_df["iteration"] = 0
    if "generator_model" not in model_df.columns:
        model_df["generator_model"] = "llama_70b"

    # 3) Concatenate base + model
    combined_df = pd.concat([base_df, model_df], ignore_index=True)
    combined_df.dropna(subset=["question_body"], inplace=True)

    # optionally filter for specific iterations
    # combined_df = combined_df[combined_df["iteration"].isin([8, 9, 10, 11, 12])]

    # Map topic => group (assumes a globally defined topic_to_group dict)
    def map_topic_to_group(raw_topic: Optional[str]) -> str:
        if not isinstance(raw_topic, str) or pd.isna(raw_topic):
            return "Other"
        return topic_to_group.get(raw_topic.lower(), "Other")

    # For iteration < 0 => label as "Base", everything else => actual group
    def label_base_or_topic(row: pd.Series) -> str:
        return "Base" if row["iteration"] < 0 else map_topic_to_group(row["topic"])

    combined_df["ColorGroup"] = combined_df.apply(label_base_or_topic, axis=1)

    # Human-friendly iteration label
    def make_label(it: int) -> str:
        return "Base" if it < 0 else f"Iteration {it}"
    combined_df["IterationLabel"] = combined_df["iteration"].apply(make_label)

    # 4) Embed
    mpnet = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
    texts = combined_df["question_body"].tolist()

    # Nicely formatted hover text
    def chunk_text_80_chars(txt: str) -> str:
        return "<br>".join([txt[i : i + 80] for i in range(0, len(txt), 80)])
    combined_df["Formatted_Question"] = [chunk_text_80_chars(tx) for tx in texts]

    embeddings = mpnet.encode(texts, show_progress_bar=True)

    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    coords = tsne.fit_transform(embeddings)
    combined_df["TSNE1"] = coords[:, 0]
    combined_df["TSNE2"] = coords[:, 1]

    # ----------------------
    # Separate base vs. non-base for different traces
    # ----------------------
    base_data = combined_df[combined_df["iteration"] < 0].copy()
    model_data = combined_df[combined_df["iteration"] >= 0].copy()

    # Plot the model data with color=ColorGroup (categorical)
    fig = px.scatter(
        model_data,
        x="TSNE1",
        y="TSNE2",
        color="ColorGroup",
        hover_data={
            "TSNE1": False,
            "TSNE2": False,
            "IterationLabel": True,
            "ColorGroup": True,
            "Formatted_Question": True
        },
        width=1100,
        height=600,
    )

    # Add base data points as an extra Scatter so they're always light grey
    fig.add_trace(
        go.Scatter(
            x=base_data["TSNE1"],
            y=base_data["TSNE2"],
            mode="markers",
            marker=dict(color="rgba(160,160,160,0.3)", size=10),
            name="Initial Static Questions",
            text=base_data["Formatted_Question"],
            hoverinfo="text",
            showlegend=False
            # showlegend=True
        )
    )

    # ----------------------
    # Annotate each topic group with its medoid (model_data only)
    # ----------------------
    groups = model_data["ColorGroup"].unique()
    for grp in groups:
        if grp == "Base" or grp == "Other":
            continue  # skip labeling base/other
        subset_df = model_data[model_data["ColorGroup"] == grp]
        if subset_df.empty:
            continue

        coords_g = subset_df[["TSNE1", "TSNE2"]].values
        dists = pairwise_distances(coords_g, metric="euclidean")
        sum_dists = dists.sum(axis=1)
        medoid_idx = np.argmin(sum_dists)
        medoid_x, medoid_y = coords_g[medoid_idx]

        fig.add_annotation(
            x=medoid_x,
            y=medoid_y,
            text=grp,
            showarrow=True,
            arrowhead=0,  # No arrowhead for a line
            arrowwidth=4,  # Increased line thickness
            arrowcolor="rgba(136,136,136,0.5)",  # Lighter back line color with 0.5 alpha
            ax=-40 if grp == "Politics & Econ" else (  # Push Politics & Econ left
                -40 if grp == "Energy & Environ" else
                -40 if grp == "AI & CS" else
                140 if grp == "Manufacturing" else  # Push Manufacturing slightly left
                60 if grp == "Media" else 0  # Push Media more right
            ),  
            ay=70 if grp == "Politics & Econ" else (
                -65 if grp == "Energy & Environ" else  # Push Energy up more
                -55 if grp == "Ag & Biotech" else  # Push Agriculture up
                -45 if grp == "Manufacturing" else  # Push Manufacturing slightly down
                -45
            ),
            font=dict(
                size=24,  # Increased font size
                color="#222",  # Darker text color
                family="Arial",  # Clean font
                weight="bold"  # Bold text
            ),
            bgcolor="white",
            bordercolor="#c7c7c7",
            borderwidth=2,  # Thicker border
            borderpad=6,  # More padding
            opacity=0.9,  # Slightly more opaque
        )
    # Layout settings
    fig.update_layout(
        plot_bgcolor="white",
        # legend=dict(
        #     yanchor="top",
        #     y=0.99,
        #     xanchor="right",
        #     x=0.99,
        #     font=dict(size=20)
        # ),
        font=dict(size=20),
        margin=dict(t=20),  # Added top margin to make room for legend
        xaxis=dict(showticklabels=False, showgrid=True),  # Remove x-axis ticks and grid
        yaxis=dict(showticklabels=False, showgrid=True)   # Remove y-axis ticks and grid
    )
    fig.update_traces(marker=dict(size=10), showlegend=False)

    fig.show()

    # ----------------------
    # Print distribution
    # ----------------------
    print("\nTopic distribution by iteration label:")
    for iteration_label in sorted(combined_df["IterationLabel"].unique()):
        subset = combined_df[combined_df["IterationLabel"] == iteration_label]
        topic_dist = subset["ColorGroup"].value_counts()
        print(f"\n{iteration_label} distribution:")
        print(topic_dist.to_string())

    return combined_df

combined_df = analyze_model_topics()

Batches:   0%|          | 0/15 [00:00<?, ?it/s]


Topic distribution by iteration label:

Base distribution:
ColorGroup
Base    100

Iteration 1 distribution:
ColorGroup
Sports       15
Tech & AI    15

Iteration 10 distribution:
ColorGroup
Tech & AI           6
Ag & Biotech        6
Media               6
Energy & Environ    6
Manufacturing       6

Iteration 11 distribution:
ColorGroup
Tech & AI           12
Ag & Biotech         6
Energy & Environ     6
Media                6

Iteration 12 distribution:
ColorGroup
Tech & AI           12
Energy & Environ     6
Ag & Biotech         6
Media                6

Iteration 2 distribution:
ColorGroup
Politics & Econ     12
Tech & AI            6
Media                6
Energy & Environ     6

Iteration 3 distribution:
ColorGroup
Tech & AI           12
Politics & Econ     12
Energy & Environ     6

Iteration 4 distribution:
ColorGroup
Tech & AI           6
Ag & Biotech        6
Energy & Environ    6
Politics & Econ.    6
Manufacturing       6

Iteration 5 distribution:
ColorGroup
Tech & AI    

In [86]:
import os
import pandas as pd
import numpy as np
from typing import Optional
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.manifold import TSNE
from sklearn.metrics import pairwise_distances
import plotly.express as px
import plotly.graph_objects as go


def analyze_model_topics_with_cache(
    csv_path: str = "llama_70b/meta_llama_Meta_Llama_3.1_70B_Instruct_Turbo_20250126_212255.csv",
    cache_dir: str = "tsne_cache",
    force_recompute: bool = False
) -> pd.DataFrame:
    """
    Combines a base dataset (iteration < 0, plotted in very light grey) with a model CSV (iteration >= 0).
    We color the scatter plot by topic groups, then compute TSNE embeddings. If a cached TSNE result
    exists for the provided csv_path, load it unless force_recompute is True. Otherwise,
    compute and store a cached CSV in cache_dir.

    Returns:
        A pandas DataFrame containing TSNE embeddings and other columns.
    """
    # Ensure cache directory exists
    os.makedirs(cache_dir, exist_ok=True)
    base_name = os.path.splitext(os.path.basename(csv_path))[0]
    cache_file = os.path.join(cache_dir, f"{base_name}_tsne_cache.csv")

    # If not forcing recomputation, attempt to load from cache
    if not force_recompute and os.path.isfile(cache_file):
        print(f"Loading TSNE data from cache: {cache_file}")
        return pd.read_csv(cache_file)

    print("Cache not found or force_recompute is True. Computing TSNE...")

    # Topic -> Group mapping (the example dict from the snippet)
    topic_to_group = {
        # Energy & Environment
        'energy storage': 'Energy & Environ',
        'economics and energy': 'Energy & Environ',
        'energy': 'Energy & Environ',
        'energy and environment': 'Energy & Environ',
        'energy policy and sustainability': 'Energy & Environ',
        'environment and sustainability': 'Energy & Environ',
        'environment and business': 'Energy & Environ',
        'environment and technology': 'Energy & Environ',
        'environmental impact and technology': 'Energy & Environ',
        'environmental sustainability and tech industry': 'Energy & Environ',
        'environment and operations': 'Energy & Environ',
        'electric vehicle manufacturing': 'Energy & Environ',
        'climate change and food security': 'Energy & Environ',
        'renewable energy': 'Energy & Environ',
        'environmental factors and tech industry': 'Energy & Environ',

        # AI & Computer Science
        'technology and regulation': 'Tech & AI',
        'materials science and quantum computing': 'Tech & AI',
        'cryptography': 'Tech & AI',
        'ai and computing': 'Tech & AI',
        'artificial general intelligence': 'Tech & AI',
        'robotics': 'Tech & AI',
        'ai and robotics': 'Tech & AI',
        'cybersecurity': 'Tech & AI',
        'artificial intelligence and quantum computing': 'Tech & AI',
        'quantum computing and ai': 'Tech & AI',

        # Media
        'social media and politics': 'Media',
        'social media and us presidential election': 'Media',
        'social media': 'Media',
        'social media and regulation': 'Media',
        'social media and mental health': 'Media',
        'election misinformation': 'Media',
        'social media regulation': 'Media',
        'marketing and e-commerce': 'Media',
        'elections and misinformation': 'Media',
        'marketing and social media': 'Media',
        'social media and electric vehicles': 'Media',
        'social media and e-commerce': 'Media',
        'social media and sales': 'Media',

        # Transportation
        'autonomous vehicles and transportation': 'Manufacturing',
        'transportation and technology': 'Manufacturing',
        'transportation and infrastructure': 'Manufacturing',
        'transportation and autonomous vehicles': 'Manufacturing',
        'autonomous vehicles': 'Manufacturing',
        'autonomous vehicles and traffic accidents': 'Manufacturing',

        # Agriculture & Biotechnology
        'agriculture and biotechnology': 'Ag & Biotech',
        'agriculture and genetic engineering': 'Ag & Biotech',
        'genetic engineering': 'Ag & Biotech',

        # Sports
        'sports': 'Sports',
        'sports and olympics': 'Sports',

        # Global Events / Politics / Economics
        'global events and supply chains': 'Politics & Econ',
        'economics': 'Politics & Econ',
        'economics and finance': 'Politics & Econ',
        'economics and technology': 'Politics & Econ',
        'economics and education': 'Politics & Econ',
        'economic sanctions and international trade': 'Politics & Econ',
        'geopolitics': 'Politics & Econ',
        'geopolitics and national security': 'Politics & Econ.',
        'politics and social media': 'Politics & Econ.',
        'international trade and economic sanctions': 'Politics & Econ.',

    }

    def map_topic_to_group(raw_topic: Optional[str]) -> str:
        if not isinstance(raw_topic, str) or pd.isna(raw_topic):
            return "Other"
        return topic_to_group.get(raw_topic.lower(), "Other")

    # 1) Load the base dataset; mark iteration < 0
    ds_base = load_dataset("prithvi3/cond_100_test")
    base_test = ds_base["test"]
    base_questions = []
    for idx, item in enumerate(base_test):
        base_questions.append({
            "question_body": item["P_and_Q_body"],
            "topic": None,  # No explicit topic for base
            "iteration": -1,
            "generator_model": "Base"
        })
    base_df = pd.DataFrame(base_questions)

    # 2) Load CSV data
    model_df = pd.read_csv(csv_path).dropna(subset=["question_body"]).copy()
    if "iteration" not in model_df.columns:
        model_df["iteration"] = 0
    if "generator_model" not in model_df.columns:
        model_df["generator_model"] = "UnknownModel"

    # 3) Concatenate base + model
    combined_df = pd.concat([base_df, model_df], ignore_index=True)
    combined_df.dropna(subset=["question_body"], inplace=True)

    # For iteration < 0 => label as "Base", everything else => actual group
    def label_base_or_topic(row: pd.Series) -> str:
        return "Base" if row["iteration"] < 0 else map_topic_to_group(row["topic"])

    combined_df["ColorGroup"] = combined_df.apply(label_base_or_topic, axis=1)

    # Human-friendly iteration label
    def make_label(it: int) -> str:
        return "Base" if it < 0 else f"Iteration {it}"
    combined_df["IterationLabel"] = combined_df["iteration"].apply(make_label)

    # 4) Embed
    mpnet = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
    texts = combined_df["question_body"].tolist()

    def chunk_text_80_chars(txt: str) -> str:
        return "<br>".join([txt[i: i + 80] for i in range(0, len(txt), 80)])
    combined_df["Formatted_Question"] = [chunk_text_80_chars(tx) for tx in texts]

    embeddings = mpnet.encode(texts, show_progress_bar=True)

    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    coords = tsne.fit_transform(embeddings)
    combined_df["TSNE1"] = coords[:, 0]
    combined_df["TSNE2"] = coords[:, 1]

    # Write out the combined_df to cache
    print(f"Saving TSNE data to cache: {cache_file}")
    combined_df.to_csv(cache_file, index=False)

    return combined_df

combined_df = analyze_model_topics_with_cache()

Loading TSNE data from cache: tsne_cache/meta_llama_Meta_Llama_3.1_70B_Instruct_Turbo_20250126_212255_tsne_cache.csv


In [99]:
def create_lplot(combined_df: pd.DataFrame) -> go.Figure:
    """
    Given a DataFrame containing TSNE embeddings, topic groups, and iteration labels,
    create a scatter plot showing model data colored by topic group, with base data shown
    in semi-transparent gray, and topic medoids labeled.
    
    Args:
        combined_df (pd.DataFrame): A DataFrame containing columns:
            - "TSNE1", "TSNE2" : Coordinates from TSNE.
            - "iteration" : Numeric iteration value.
            - "ColorGroup" : Assigned grouping (e.g., "Base", or any topic group).
            - "Formatted_Question" : Pre-formatted text for hover.
            - "IterationLabel" : String label for iteration.
    
    Returns:
        go.Figure: The Plotly figure.
    """
    import numpy as np
    from sklearn.metrics import pairwise_distances
    import plotly.express as px
    import plotly.graph_objects as go

    # Separate base vs. model for layered plotting
    base_data = combined_df[combined_df["iteration"] < 0].copy()
    model_data = combined_df[combined_df["iteration"] >= 0].copy()

    # Main scatter: model data colored by ColorGroup
    fig = px.scatter(
        model_data,
        x="TSNE1",
        y="TSNE2",
        color="ColorGroup",
        hover_data={
            "TSNE1": False,
            "TSNE2": False,
            "IterationLabel": True,
            "ColorGroup": True,
            "Formatted_Question": True
        },
        width=1100,
        height=600,
    )

    # Add base data in light gray
    fig.add_trace(
        go.Scatter(
            x=base_data["TSNE1"],
            y=base_data["TSNE2"],
            mode="markers",
            marker=dict(color="rgba(160,160,160,0.3)", size=10),
            name="Initial Static Questions",
            text=base_data["Formatted_Question"],
            hoverinfo="text",
            showlegend=False,
        )
    )

    # Label each topic group with its medoid, skipping Base or Other
    groups = model_data["ColorGroup"].unique()
    for grp in groups:
        if grp == "Base" or grp == "Other":
            continue
        subset_df = model_data[model_data["ColorGroup"] == grp]
        if subset_df.empty:
            continue

        coords_g = subset_df[["TSNE1", "TSNE2"]].values
        dists = pairwise_distances(coords_g, metric="euclidean")
        sum_dists = dists.sum(axis=1)
        medoid_idx = np.argmin(sum_dists)
        medoid_x, medoid_y = coords_g[medoid_idx]

        fig.add_annotation(
            x=medoid_x,
            y=medoid_y,
            text=grp,
            showarrow=True,
            arrowhead=0,
            arrowwidth=4,
            arrowcolor="rgba(136,136,136,0.5)",
            ax=40 if grp == "Politics & Econ." else (
                -40 if grp == "Energy & Environ" else
                -40 if grp == "AI & CS" else
                140 if grp == "Manufacturing" else
                60 if grp == "Media" else 0  
            ),
            ay=30 if grp == "Politics & Econ." else (
                -65 if grp == "Energy & Environ" else
                -55 if grp == "Ag & Biotech" else
                -45 if grp == "Manufacturing" else
                -45
            ),
            font=dict(
                size=24,
                color="#222",
                family="Arial",
                weight="bold"
            ),
            bgcolor="white",
            bordercolor="#c7c7c7",
            borderwidth=2,
            borderpad=6,
            opacity=0.9,
        )

    # Style updates
    fig.update_layout(
        plot_bgcolor="white",
        font=dict(size=20),
        margin=dict(t=20),
        xaxis=dict(showticklabels=False, showgrid=True),
        yaxis=dict(showticklabels=False, showgrid=True),
    )
    fig.update_traces(marker=dict(size=10), showlegend=False)

    return fig

In [100]:
fig = create_lplot(combined_df)
fig.show()

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [ ]:
combined_df_deepseek = analyze_model_topics_with_cache(
    csv_path="/Users/davisbrown/adaptive_evals/tasks/cond_100/deepseek-v3/deepseek_chat_20250127_212346.csv"
)
fig = create_lplot(combined_df_deepseek)
fig.show()

